# Exploration — Lumina & Co (notebook personnel de Kémil)

Ce notebook est séparé de `notebook_TP1_...` et `notebook_TP2_...` pour ne pas perturber le travail des partenaires de groupe.

**Point de départ :** en observant les données, on a remarqué qu'une seule campagne — celle du printemps (`Spring_Launch`) — dépasse le million d'euros de revenu, alors que les autres non. Objectif : comprendre pourquoi, et voir ce qui est réplicable sur les autres campagnes.

**Deuxième volet :** creuser les profils clients au-delà du RFM brut, en croisant la segmentation comportementale avec les données zero-party (préférences déclarées, âge, situation de vie, densité urbaine) pour voir si ça permet d'affiner l'expérience client.


## 1. Le pic du printemps est-il un effet saisonnier récurrent ?

On recalcule le chiffre d'affaires réel par saison, année par année, à partir de `transactions_clean.csv` (donnée déjà nettoyée en TP1, avec le fix du filtre `code_atypique`).

In [3]:
import pandas as pd
transac_per_y = pd.read_csv("clean_data/transactions_clean.csv", parse_dates=["invoice_date"])

transac_per_y["annee"]  = transac_per_y["invoice_date"].dt.year
transac_per_y["mois"]   = transac_per_y["invoice_date"].dt.month
transac_per_y["saison"] = transac_per_y["mois"].map({
    12: "Hiver", 1: "Hiver", 2: "Hiver",
    3: "Printemps", 4: "Printemps", 5: "Printemps",
    6: "Ete", 7: "Ete", 8: "Ete",
    9: "Automne", 10: "Automne", 11: "Automne"
})

print("====== CA par annee x saison ======")
ca_annee_saison = transac_per_y.groupby(["annee", "saison"])["line_total"].sum().unstack().round(0)
print(ca_annee_saison)

print("\n====== CA total par saison (toutes annees confondues) ======")
print(transac_per_y.groupby("saison")["line_total"].sum().sort_values(ascending=False).round(0))


====== CA par annee x saison ======
saison    Automne        Ete      Hiver  Printemps
annee                                             
2022      17669.0     5180.0    12890.0     2260.0
2023     210884.0   133472.0   120501.0    71867.0
2024    2902837.0  2007399.0  1283575.0   528058.0
2025    3144198.0  3488312.0  3110648.0  3924023.0
2026          NaN  1354019.0  2146118.0  4176421.0

====== CA total par saison (toutes annees confondues) ======
saison
Printemps    8702629.0
Ete          6988381.0
Hiver        6673733.0
Automne      6275589.0
Name: line_total, dtype: float64


**Lecture :** en 2022, 2023 et 2024, le printemps est en réalité la saison la **plus faible** de l'année (ex: 2024 → 528k€ contre 1,28M-2,9M pour les autres saisons). Ce n'est qu'en 2025 qu'il dépasse à peine l'été, et en 2026 la comparaison est biaisée car l'été et l'automne n'ont que quelques mois de données.

**Conclusion : il n'y a pas d'effet saisonnier naturel qui favorise le printemps.** Le pic de 2026 n'est pas une récurrence historique.

## 2. Ce que dit `campaigns.csv` : budget vs efficacité

Cette table (et `touchpoints.csv`) existent dans le projet mais n'avaient jamais été exploitées en TP1/TP2. Elles contiennent la performance agrégée par campagne (budget, revenu, taux de conversion, ROAS).

In [ ]:
campaigns = pd.read_csv("Lumina & Co - CRM/campaigns.csv")

campaigns["budget_relatif"] = (campaigns["total_cost"] / campaigns["total_cost"].min()).round(2)

print("====== Campagnes triees par revenu ======")
cols = ["campaign_name", "total_cost", "revenue", "conversion_rate", "roas", "cpa", "primary_channel"]
print(campaigns.sort_values("revenue", ascending=False)[cols].to_string(index=False))

print("\n====== Budget relatif (vs la campagne la moins chere) ======")
print(campaigns[["campaign_name", "total_cost", "budget_relatif"]].sort_values("total_cost", ascending=False).to_string(index=False))

print("\n====== Simulation : et si Winter_Promo avait eu le budget de Spring_Launch ? ======")
winter = campaigns[campaigns["campaign_name"] == "Winter_Promo"].iloc[0]
spring = campaigns[campaigns["campaign_name"] == "Spring_Launch"].iloc[0]
revenu_hypothetique = spring["total_cost"] * winter["roas"]
print(f"Budget Spring_Launch applique au ROAS de Winter_Promo: {revenu_hypothetique:,.0f} EUR")
print(f"Revenu reel de Spring_Launch: {spring['revenue']:,.0f} EUR")


====== Campagnes triees par revenu ======
 campaign_name  total_cost    revenue  conversion_rate  roas   cpa primary_channel
 Spring_Launch 374813.3262 1686555.27           0.7655  4.50 18.40         display
  Black_Friday 205749.4125  957170.87           0.6971  4.65 10.96         display
     Valentine 247465.9644  853184.79           0.5857  3.45 16.34         display
Back_to_School 159256.4075  725541.50           0.5168  4.56 12.07          social
  Winter_Promo 133949.0025  716632.64           0.5527  5.35  9.45         display
   Summer_Sale 226774.1597  621009.60           0.5010  2.74 17.84         display

====== Budget relatif (vs la campagne la moins chere) ======
 campaign_name  total_cost  budget_relatif
 Spring_Launch 374813.3262            2.80
     Valentine 247465.9644            1.85
   Summer_Sale 226774.1597            1.69
  Black_Friday 205749.4125            1.54
Back_to_School 159256.4075            1.19
  Winter_Promo 133949.0025            1.00

====== Simula

**Lecture :** `Spring_Launch` a un budget de 374 813€ — **2,8× le budget de la campagne la moins chère** (`Winter_Promo`, 133 949€). C'est le plus gros budget de toutes les campagnes.

Son ROAS (retour par euro dépensé) est de 4,50 — ce n'est même pas le meilleur : `Winter_Promo` fait 5,35 avec un budget presque 3× plus petit.

**Conclusion : le printemps ne dépasse le million pas grâce à un effet magique, mais parce qu'on lui a alloué beaucoup plus de budget — et pas forcément au canal le plus efficace.** À budget égal, `Winter_Promo` aurait généré davantage de revenu que `Spring_Launch`.

**Piste actionnable pour "réitérer" :** ne pas copier le printemps, mais réallouer une partie du budget vers les campagnes les plus efficientes (ROAS le plus élevé), et comprendre pourquoi leur mix canal/timing convertit mieux par euro dépensé — à explorer avec `touchpoints.csv` (canaux, position dans le parcours) dans une prochaine étape.

## 3. Segmentation clients par règles (au-delà des 3 cas extrêmes)

La consigne TP2 demande de dépasser les segments "555 / 111 / à risque" (qui ne couvrent que les cas extrêmes) avec des règles par plage justifiées, pour arriver à 5-8 segments nommés couvrant toute la base.

In [ ]:
customers = pd.read_csv("clean_data/customers_clean.csv")

# Scores RFM par quintiles (1 a 5)
customers["score_R"] = pd.qcut(customers["recency_days"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
customers["score_F"] = pd.qcut(customers["n_orders"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
customers["score_M"] = pd.qcut(customers["total_spent"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segmenter(row):
    r, f, m = row["score_R"], row["score_F"], row["score_M"]
    if r >= 4 and f >= 4 and m >= 4:            return "Champions"
    if r >= 4 and f <= 2:                       return "Nouveaux prometteurs"
    if r <= 2 and f >= 4 and m >= 4:            return "A risque (forte valeur, inactifs)"
    if r <= 2 and f <= 2 and m <= 2:            return "Perdus"
    if r >= 3 and f >= 3 and m >= 3:            return "Fideles"
    if r <= 2:                                  return "Hibernants"
    return "Clients moyens"

customers["segment"] = customers.apply(segmenter, axis=1)

print("====== Repartition des segments ======")
rep = customers["segment"].value_counts()
print(rep)
print((rep / len(customers) * 100).round(1).astype(str) + " %")

print("\n====== Part du CA par segment ======")
ca_seg = customers.groupby("segment")["total_spent"].sum().sort_values(ascending=False)
print((ca_seg / customers["total_spent"].sum() * 100).round(1).astype(str) + " %")


====== Repartition des segments ======
segment
Champions                            10003
Perdus                                9625
Fideles                               9457
Hibernants                            8192
Clients moyens                        6228
Nouveaux prometteurs                  4089
A risque (forte valeur, inactifs)     2035
Name: count, dtype: int64
segment
Champions                            20.2 %
Perdus                               19.4 %
Fideles                              19.1 %
Hibernants                           16.5 %
Clients moyens                       12.5 %
Nouveaux prometteurs                  8.2 %
A risque (forte valeur, inactifs)     4.1 %
Name: count, dtype: str

====== Part du CA par segment ======
segment
Champions                            53.0 %
Fideles                              22.3 %
Hibernants                            9.0 %
A risque (forte valeur, inactifs)     7.0 %
Clients moyens                        3.0 %
Nouveaux prometteurs

**Règles utilisées** (documentées, à adapter/justifier dans le rendu final) :
- **Champions** : récent, fréquent, gros montant (R≥4, F≥4, M≥4)
- **Nouveaux prometteurs** : récent mais peu de commandes (R≥4, F≤2)
- **À risque** : forte valeur historique mais inactif depuis longtemps (R≤2, F≥4, M≥4)
- **Perdus** : tout est bas (R≤2, F≤2, M≤2)
- **Fidèles** : scores moyens-hauts sur les trois dimensions (R≥3, F≥3, M≥3)
- **Hibernants** : peu récent, sans forte valeur ni forte fréquence (R≤2, reste)
- **Clients moyens** : tout ce qui ne rentre dans aucune règle ci-dessus

**Lecture :** 7 segments, 100% de la base couverte. Champions = 20,2% des clients mais 53% du CA (Pareto confirmé). Perdus = 19,4% des clients pour seulement 2,8% du CA. À risque = seulement 4,1% des clients mais 7% du CA — prioritaires pour une action de réactivation ciblée.

## 4. Le comportement d'achat (RFM) est-il lié au profil déclaré (zero-party) ?

"Pour aller plus loin" du TP2 : croiser un segment RFM avec un champ zero-party pour voir si ça révèle un profil plus précis qu'avec le RFM seul.

In [ ]:
zp_cols = ["declared_preference", "age_bracket", "life_stage", "urban_density"]

print("====== Taux de remplissage zero-party PAR SEGMENT (%) ======")
taux = customers.groupby("segment")[zp_cols].apply(lambda g: g.notna().mean() * 100).round(1)
print(taux)

print("\n====== declared_preference : repartition PAR SEGMENT, parmi ceux qui ont repondu (%) ======")
sub = customers[customers["declared_preference"].notna()]
crosstab = (pd.crosstab(sub["segment"], sub["declared_preference"], normalize="index") * 100).round(1)
print(crosstab)


====== Taux de remplissage zero-party PAR SEGMENT (%) ======
                                   declared_preference  ...  urban_density
segment                                                 ...               
A risque (forte valeur, inactifs)                 37.5  ...           21.3
Champions                                         38.0  ...           20.1
Clients moyens                                    38.6  ...           19.6
Fideles                                           37.8  ...           19.7
Hibernants                                        37.5  ...           19.5
Nouveaux prometteurs                              37.9  ...           19.1
Perdus                                            39.0  ...           19.9

[7 rows x 4 columns]

====== declared_preference : repartition PAR SEGMENT, parmi ceux qui ont repondu (%) ======
declared_preference                Anti-imperfections  ...  Sans parfum
segment                                                ...             
A ris

**Lecture :** le taux de réponse aux enquêtes (37-39% pour la préférence, 42-43% pour l'âge, ~20-23% pour la situation de vie et la densité urbaine) est quasi identique quel que soit le segment — un Champion ne répond pas plus ou moins qu'un client Perdu. La répartition des préférences déclarées est également plate sur tous les segments (écarts de 1 à 3 points = bruit statistique, pas un signal).

**Conclusion : dans cette base, le comportement d'achat (RFM) et le profil déclaré (zero-party) sont statistiquement indépendants.** Croiser les deux n'affine pas la segmentation — mais le zero-party reste utile *au niveau individuel* : personnaliser le message ou le produit recommandé à un client qui a répondu à l'enquête, indépendamment de son segment de valeur.


## Synthèse

1. Le pic de revenu du printemps 2026 n'est pas un effet saisonnier récurrent (il était même historiquement la saison la plus faible) — c'est un effet de budget marketing (2,8× le budget de la campagne la moins chère), pas d'efficacité (ROAS pas le meilleur).
2. Piste d'action : réallouer du budget vers les campagnes à meilleur ROAS (`Winter_Promo`) plutôt que de chercher à "reproduire" le printemps.
3. La segmentation RFM par règles couvre 100% de la base en 7 segments actionnables, avec une forte concentration de valeur sur les Champions (53% du CA pour 20% des clients).
4. Le croisement RFM × zero-party ne révèle pas de profil démographique/préférence distinctif par segment — ces données restent utiles pour la personnalisation individuelle, pas pour la segmentation.

**Prochaine étape possible :** creuser `touchpoints.csv` (canal, position dans le parcours, coût par touchpoint) pour comprendre precisement *pourquoi* `Winter_Promo` convertit mieux par euro dépensé que les autres campagnes.
